In [1]:
"""Build outputs for the new architecture.

New architecture composition:
  1. optional ViHealthBERT/Hugging Face NER semantic seeds
  2. specialized Drug parser using dictionary/RxNorm/NER seeds
  3. specialized Lab parser using curated dictionary/NER seeds
  4. dictionary/rule modules for diagnosis, symptoms, structural fallback
  5. assertion, merge, ICD/RxNorm linking, schema validation

The script can run without NER by omitting --ner-model. In that mode it still
exercises Drug parser + Lab parser + Dictionary/Rules modules and writes outputs
that can be compared with the old V0/V0-linked outputs.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from collections import Counter
from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple

ROOT = "/Users/mac/Dev/Viettel_AI_Race/ViClinicalIE"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.assertion import add_assertions
from src.drug_parser import parse_drug_candidates
from src.io_utils import load_input_files
from src.lab_parser import load_lab_dictionary, parse_lab_candidates
from src.linking.candidate_linker import link_mapping_candidates, load_default_linkers
from src.merge import merge_candidates
from src.models import ClinicalDocument, SpanCandidate
from src.output_writer import create_output_zip, write_output_files
from src.rule_extractors import (
    ENTITY_DIAGNOSIS,
    ENTITY_DRUG,
    ENTITY_LAB_NAME,
    ENTITY_LAB_RESULT,
    ENTITY_SYMPTOM,
    TARGET_ENTITY_TYPES,
    dedupe_candidates,
    extract_diagnosis_candidates,
    extract_structural_candidates,
    extract_symptom_candidates,
    extraction_summary,
    read_term_csv,
    reject_non_target_candidates,
    validate_candidate_offsets,
    write_span_candidates_jsonl,
)
from src.section_parser import parse_documents
from src.validator import validate_output_artifacts, write_validation_report
from src.vihealthbert_ner import VIETMED_DEFAULT_THRESHOLDS, FastTokenizerRequiredError, HuggingFaceTokenPredictor, ViHealthBERTNER


LAB_TYPES = {ENTITY_LAB_NAME, ENTITY_LAB_RESULT}
MAPPING_TYPES = {ENTITY_DIAGNOSIS, ENTITY_DRUG}

In [ ]:
from pathlib import Path
from pprint import pprint

ROOT = Path("/Users/mac/Dev/Viettel_AI_Race/ViClinicalIE")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.io_utils import load_input_files
from src.section_parser import parse_documents
from src.vihealthbert_ner import (
    HuggingFaceTokenPredictor,
    ViHealthBERTNER,
    VIETMED_DEFAULT_THRESHOLDS,
    map_vietmed_label,
    route_diseasesyptom_candidates,
    deduplicate_ner_candidates,
    decode_token_predictions,
)
from src.models import ClinicalDocument
from src.preprocessing import TextWindow

# -----------------------------
# Config
# -----------------------------
INPUT_DIR = ROOT / "input"
MODEL_NAME = None  # set to your HF model id or local checkpoint path
LABEL_MAP = "compact"  # or "vietmed"
DEVICE = None
MAX_LENGTH = 512

# Optional thresholds
THRESHOLDS = {
    # "THUỐC": 0.75,
    # "TÊN_XÉT_NGHIỆM": 0.70,
    # "KẾT_QUẢ_XÉT_NGHIỆM": 0.80,
    # "CHẨN_ĐOÁN": 0.80,
    # "TRIỆU_CHỨNG": 0.80,
}

# -----------------------------
# Load one document
# -----------------------------
docs = parse_documents(load_input_files(str(INPUT_DIR)))
doc = sorted(docs, key=lambda d: int(d.file_id) if d.file_id.isdigit() else 10**9)[0]

print("FILE ID:", doc.file_id)
print("TEXT LEN:", len(doc.raw_text))
print("WINDOWS:", len(doc.model_windows or doc.sentence_windows))
print("LINES:", len(doc.lines))

windows = doc.model_windows or doc.sentence_windows
if not windows and doc.raw_text:
    windows = [TextWindow(doc.raw_text, 0, len(doc.raw_text), 0, "model")]

# -----------------------------
# Initialize model backend
# -----------------------------
predictor = HuggingFaceTokenPredictor(
    MODEL_NAME,
    device=DEVICE,
    max_length=MAX_LENGTH,
    label_map=LABEL_MAP,
)

ner = ViHealthBERTNER(
    predictor,
    thresholds=THRESHOLDS if THRESHOLDS else (VIETMED_DEFAULT_THRESHOLDS if LABEL_MAP == "vietmed" else {}),
    label_map=LABEL_MAP,
)

print("\nMODEL READY")
print("LABEL MAP:", LABEL_MAP)
print("THRESHOLDS:", ner.thresholds)

# -----------------------------
# Inspect each window
# -----------------------------
all_candidates = []

for i, window in enumerate(windows[:5]):  # limit for debugging
    print("\n" + "=" * 100)
    print(f"WINDOW {i}")
    print("WINDOW ID:", window.window_id)
    print("SPAN:", (window.start, window.end))
    print("TEXT:", repr(window.text[:300]))

    preds = predictor(window.text)
    print(f"\nRAW TOKENS ({len(preds)}):")
    for p in preds[:40]:
        print(f"  {p.label:30} {p.start:>4}-{p.end:<4} conf={p.confidence:.4f} text={repr(window.text[p.start:p.end])}")

    if LABEL_MAP == "vietmed":
        mapped = [type(p)(label=map_vietmed_label(p.label), start=p.start, end=p.end, confidence=p.confidence) for p in preds]
        routed_preview = route_diseasesyptom_candidates(
            decode_token_predictions(
                doc.raw_text,
                doc.file_id,
                mapped,
                offset_base=window.start,
                source="debug_ner",
                window_id=window.window_id,
            ),
            doc.lines,
        )
        candidates = routed_preview
    else:
        candidates = decode_token_predictions(
            doc.raw_text,
            doc.file_id,
            preds,
            offset_base=window.start,
            source="debug_ner",
            window_id=window.window_id,
        )

    print(f"\nDECODED CANDIDATES ({len(candidates)}):")
    for c in candidates[:40]:
        print(
            f"  type={c.type_candidate:20} "
            f"span=({c.start},{c.end}) "
            f"conf={c.confidence:.4f} "
            f"text={repr(c.text)} "
            f"source={c.source}"
        )

    all_candidates.extend(candidates)

# -----------------------------
# Final filtering + dedup
# -----------------------------
filtered = [
    c for c in all_candidates
    if c.confidence >= ner.thresholds.get(c.type_candidate, 0.0)
    and doc.raw_text[c.start:c.end] == c.text
]
final_candidates = deduplicate_ner_candidates(filtered)

print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("RAW CANDIDATES:", len(all_candidates))
print("FILTERED CANDIDATES:", len(filtered))
print("FINAL DEDUPED:", len(final_candidates))

for c in final_candidates[:100]:
    print(
        f"{c.type_candidate:20} "
        f"{c.start:>5}-{c.end:<5} "
        f"conf={c.confidence:.4f} "
        f"{repr(c.text)}"
    )


TypeError: unsupported operand type(s) for /: 'str' and 'str'